## 🎯 Project Goal
Build a **machine learning classification model** to predict whether an online purchase order 
is at **high risk (yes)** or **low risk (no)** of payment default.

- Dataset: 30,000 online purchase orders with 44 attributes
- Target column: `CLASS` (yes = high risk, no = low risk)
- Constraint: No neural networks or deep learning — traditional ML only
- Performance target: **70%+ F1-score on the "yes" (high-risk) class**

---

## 📋 Project Workflow
1. Load and explore the raw data
2. Clean and preprocess the data
3. Handle class imbalance
4. Build and evaluate the model
5. Interpret and report results

## Step 1: Import Libraries
We import all the libraries needed for this project upfront:
- `pandas` — for loading and manipulating the dataset
- `numpy` — for numerical operations
- `matplotlib` — for visualizations
- `sklearn` — for model building, splitting, and evaluation
- `imblearn` — for handling class imbalance using SMOTE

In [1]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import classification_report, f1_score
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split  
from sklearn.preprocessing import StandardScaler


 ## Step 2: Load the Dataset
The raw dataset is a tab-separated file (`risk-train.txt`) with 30,000 orders and 44 attributes.
- Separator: `\t` (tab)
- Missing values are represented as `?` in the raw file — we tell pandas to treat them as `NaN`
- `NaN` means "Not a Number" — it's pandas' way of representing missing/unknown values

In [2]:
df = pd.read_csv('risk-train.txt', sep='\t', na_values='?')

In [3]:
print(df.head())
print(df.shape)
print(df.info())

   ORDER_ID CLASS B_EMAIL B_TELEFON B_BIRTHDATE FLAG_LRIDENTISCH  \
0     49917    no     yes        no   1/17/1973              yes   
1     49919    no     yes       yes   12/8/1970               no   
2     49923    no     yes        no    4/3/1972              yes   
3     49924    no      no       yes    8/1/1966              yes   
4     49927    no     yes       yes  12/21/1969              yes   

  FLAG_NEWSLETTER    Z_METHODE Z_CARD_ART  Z_CARD_VALID  ... FAIL_RPLZ  \
0             yes        check        NaN        5.2006  ...        no   
1              no  credit_card       Visa       12.2007  ...       yes   
2              no        check        NaN       12.2007  ...        no   
3              no        check        NaN        1.2007  ...        no   
4              no  credit_card   Eurocard       12.2006  ...        no   

   FAIL_RORT FAIL_RPLZORTMATCH SESSION_TIME  NEUKUNDE  AMOUNT_ORDER_PRE  \
0         no                no            8       yes                 0

## Step 3: Drop Irrelevant Columns
We remove columns that are not useful for the model:
- `ORDER_ID` — just a unique identifier, carries no predictive value
- `Z_LAST_NAME` — personal information, not relevant + too many missing values
- `ANUMMER_10` — completely empty, 100% missing values (30,000 nulls)

In [4]:
df = df.drop(columns=['ORDER_ID', 'Z_LAST_NAME', 'ANUMMER_10'])

In [5]:
print(df.shape)
print(df.columns.tolist())

(30000, 41)
['CLASS', 'B_EMAIL', 'B_TELEFON', 'B_BIRTHDATE', 'FLAG_LRIDENTISCH', 'FLAG_NEWSLETTER', 'Z_METHODE', 'Z_CARD_ART', 'Z_CARD_VALID', 'VALUE_ORDER', 'WEEKDAY_ORDER', 'TIME_ORDER', 'AMOUNT_ORDER', 'ANUMMER_01', 'ANUMMER_02', 'ANUMMER_03', 'ANUMMER_04', 'ANUMMER_05', 'ANUMMER_06', 'ANUMMER_07', 'ANUMMER_08', 'ANUMMER_09', 'CHK_LADR', 'CHK_RADR', 'CHK_KTO', 'CHK_CARD', 'CHK_COOKIE', 'CHK_IP', 'FAIL_LPLZ', 'FAIL_LORT', 'FAIL_LPLZORTMATCH', 'FAIL_RPLZ', 'FAIL_RORT', 'FAIL_RPLZORTMATCH', 'SESSION_TIME', 'NEUKUNDE', 'AMOUNT_ORDER_PRE', 'VALUE_ORDER_PRE', 'DATE_LORDER', 'MAHN_AKT', 'MAHN_HOECHST']


## Step 4: Handle Missing Values - Numeric Columns
Numeric columns with missing values are filled with the **median** of each column.
We use median instead of mean because median is resistant to outliers.
For example, one extremely large order value won't skew the median the way it would the mean.

Columns filled with median:
- `ANUMMER_02` to `ANUMMER_09` — order-related numeric features
- `MAHN_AKT` and `MAHN_HOECHST` — payment reminder related features

In [6]:
numeric_cols = ['ANUMMER_02', 'ANUMMER_03', 'ANUMMER_04', 'ANUMMER_05',
                'ANUMMER_06', 'ANUMMER_07', 'ANUMMER_08', 'ANUMMER_09',
                'MAHN_AKT', 'MAHN_HOECHST']

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

In [7]:
print(df[['ANUMMER_02', 'ANUMMER_03', 'MAHN_AKT', 'MAHN_HOECHST']].isnull().sum())


ANUMMER_02      0
ANUMMER_03      0
MAHN_AKT        0
MAHN_HOECHST    0
dtype: int64


## Step 5: Feature Engineering - B_BIRTHDATE → AGE
Rather than filling missing birthdates directly, we convert the column 
into a more meaningful feature: the customer's AGE.

- Convert `B_BIRTHDATE` to datetime format
- Calculate AGE = 2026 - birth year
- Fill any missing AGE values with the median age
- Drop the original `B_BIRTHDATE` column (no longer needed)

In [8]:
df['B_BIRTHDATE'] = pd.to_datetime(df['B_BIRTHDATE'], errors='coerce')
df['AGE'] = 2026 - df['B_BIRTHDATE'].dt.year
df['AGE'] = df['AGE'].fillna(df['AGE'].median())
df = df.drop(columns=['B_BIRTHDATE'])

In [9]:
print(df['AGE'].isnull().sum())
print(df['AGE'].describe())

0
count    30000.000000
mean        55.278233
std          9.158221
min         40.000000
25%         49.000000
50%         54.000000
75%         61.000000
max         92.000000
Name: AGE, dtype: float64


## Step 6: Handle Missing Values - Z_CARD_ART (Categorical)
`Z_CARD_ART` is a categorical column (card type) with 18,654 missing values.
We cannot use median or mean on text categories, so we use **mode** instead.

**Mode** = the most frequently occurring category in the column.
For example, if "Visa" appears most often, we fill all missing values with "Visa".

In [10]:
df['Z_CARD_ART'] = df['Z_CARD_ART'].fillna(df['Z_CARD_ART'].mode()[0])

In [11]:
print(df['Z_CARD_ART'].isnull().sum())
print(df['Z_CARD_ART'].value_counts())

0
Z_CARD_ART
Eurocard      23750
Visa           3927
debit_card     1550
Amex            773
Name: count, dtype: int64


## Step 7: Feature Engineering - TIME_ORDER → ORDER_HOUR & ORDER_MINUTE
`TIME_ORDER` stores the time an order was placed (e.g., "9:13", "17:36").
Raw time strings are not useful for a model, so we extract two numeric features:
- `ORDER_HOUR` — the hour the order was placed (0-23)
- `ORDER_MINUTE` — the minute the order was placed (0-59)

This way the model can learn patterns like:
"Orders placed late at night may have higher default risk"

In [12]:
df['TIME_ORDER'] = pd.to_datetime(df['TIME_ORDER'], format='%H:%M', errors='coerce')
df['ORDER_HOUR'] = df['TIME_ORDER'].dt.hour
df['ORDER_MINUTE'] = df['TIME_ORDER'].dt.minute
df['ORDER_HOUR'] = df['ORDER_HOUR'].fillna(df['ORDER_HOUR'].median())
df['ORDER_MINUTE'] = df['ORDER_MINUTE'].fillna(df['ORDER_MINUTE'].median())
df = df.drop(columns=['TIME_ORDER'])

In [13]:
print(df['ORDER_HOUR'].isnull().sum())
print(df['ORDER_MINUTE'].isnull().sum())
print(df['ORDER_HOUR'].value_counts().head())

0
0
ORDER_HOUR
14.0    1896
11.0    1872
15.0    1726
10.0    1672
13.0    1671
Name: count, dtype: int64


### 📊 Observation: Order Hours Pattern
Most orders are placed during business hours (10am - 3pm), 
with 2pm being the peak hour. This suggests customers shop 
during lunch breaks or afternoon work hours.

This feature could be valuable for the model — orders placed 
at unusual hours (e.g., 2am - 5am) may correlate with higher 
default risk.

## Step 8: Feature Engineering - DATE_LORDER → Last Order Date Parts
`DATE_LORDER` stores the date of the customer's last order (e.g., "5/12/2002").
Similar to TIME_ORDER, raw dates are not useful for a model.
We extract three numeric features:
- `LAST_ORDER_YEAR` — year of last order
- `LAST_ORDER_MONTH` — month of last order
- `LAST_ORDER_DAY` — day of last order

This helps the model learn patterns like:
"Customers who haven't ordered in many years may be higher risk"

In [14]:
df['DATE_LORDER'] = pd.to_datetime(df['DATE_LORDER'], format='%m/%d/%Y', errors='coerce')
df['LAST_ORDER_YEAR'] = df['DATE_LORDER'].dt.year
df['LAST_ORDER_MONTH'] = df['DATE_LORDER'].dt.month
df['LAST_ORDER_DAY'] = df['DATE_LORDER'].dt.day
df['LAST_ORDER_YEAR'] = df['LAST_ORDER_YEAR'].fillna(df['LAST_ORDER_YEAR'].median())
df['LAST_ORDER_MONTH'] = df['LAST_ORDER_MONTH'].fillna(df['LAST_ORDER_MONTH'].median())
df['LAST_ORDER_DAY'] = df['LAST_ORDER_DAY'].fillna(df['LAST_ORDER_DAY'].median())
df = df.drop(columns=['DATE_LORDER'])

In [15]:
print(df['LAST_ORDER_YEAR'].isnull().sum())
print(df['LAST_ORDER_MONTH'].isnull().sum())
print(df['LAST_ORDER_DAY'].isnull().sum())
print(df['LAST_ORDER_YEAR'].describe())

0
0
0
count    30000.000000
mean      2002.764567
std          0.847011
min       2000.000000
25%       2003.000000
50%       2003.000000
75%       2003.000000
max       2005.000000
Name: LAST_ORDER_YEAR, dtype: float64


### 📊 Observation: Last Order Date Pattern
All last order dates fall between 2000-2005, suggesting this is 
historical data. Customers with older last order dates may represent 
inactive customers, which could correlate with higher default risk.

## Step 9: Encode Categorical Columns
Machine learning models cannot work with text categories directly.
We need to convert them into numbers using **One-Hot Encoding**.

**One-Hot Encoding** creates a new binary column (0 or 1) for each category.
For example, the `Z_CARD_ART` column with values [Visa, Eurocard, Amex] becomes:
- `Z_CARD_ART_Visa`     → 1 if Visa, 0 otherwise
- `Z_CARD_ART_Eurocard` → 1 if Eurocard, 0 otherwise
- `Z_CARD_ART_Amex`     → 1 if Amex, 0 otherwise

We use `drop_first=True` to avoid the **dummy variable trap** — 
if we know all other categories are 0, the last one is implied.

In [16]:
X = pd.get_dummies(df.drop(columns=['CLASS']), drop_first=True)
y = df['CLASS'].map({'no': 0, 'yes': 1})

In [17]:
# Verify encoding results - check shape, columns and class distribution
print(X.shape)
print(X.columns.tolist())
print(y.value_counts())

(30000, 52)
['Z_CARD_VALID', 'VALUE_ORDER', 'AMOUNT_ORDER', 'ANUMMER_01', 'ANUMMER_02', 'ANUMMER_03', 'ANUMMER_04', 'ANUMMER_05', 'ANUMMER_06', 'ANUMMER_07', 'ANUMMER_08', 'ANUMMER_09', 'SESSION_TIME', 'AMOUNT_ORDER_PRE', 'VALUE_ORDER_PRE', 'MAHN_AKT', 'MAHN_HOECHST', 'AGE', 'ORDER_HOUR', 'ORDER_MINUTE', 'LAST_ORDER_YEAR', 'LAST_ORDER_MONTH', 'LAST_ORDER_DAY', 'B_EMAIL_yes', 'B_TELEFON_yes', 'FLAG_LRIDENTISCH_yes', 'FLAG_NEWSLETTER_yes', 'Z_METHODE_credit_card', 'Z_METHODE_debit_card', 'Z_METHODE_debit_note', 'Z_CARD_ART_Eurocard', 'Z_CARD_ART_Visa', 'Z_CARD_ART_debit_card', 'WEEKDAY_ORDER_Monday', 'WEEKDAY_ORDER_Saturday', 'WEEKDAY_ORDER_Sunday', 'WEEKDAY_ORDER_Thursday', 'WEEKDAY_ORDER_Tuesday', 'WEEKDAY_ORDER_Wednesday', 'CHK_LADR_yes', 'CHK_RADR_yes', 'CHK_KTO_yes', 'CHK_CARD_yes', 'CHK_COOKIE_yes', 'CHK_IP_yes', 'FAIL_LPLZ_yes', 'FAIL_LORT_yes', 'FAIL_LPLZORTMATCH_yes', 'FAIL_RPLZ_yes', 'FAIL_RORT_yes', 'FAIL_RPLZORTMATCH_yes', 'NEUKUNDE_yes']
CLASS
0    28254
1     1746
Name: cou

## Step 10: Train/Test Split
We split the data into:
- **80% training** — used to teach the model
- **20% testing** — used to evaluate the model on unseen data

Key parameter: `stratify=y` ensures both splits maintain the 
same class ratio (94% no, 6% yes) as the original dataset.
Without stratify, the test set might accidentally have very few 
"yes" samples, making evaluation unreliable.

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [19]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:", y_train.value_counts().to_dict())
print("y_test distribution:", y_test.value_counts().to_dict())

X_train shape: (24000, 52)
X_test shape: (6000, 52)
y_train distribution: {0: 22603, 1: 1397}
y_test distribution: {0: 5651, 1: 349}


## Step 11: Handle Class Imbalance with SMOTE
Our dataset is heavily imbalanced:
- 94.2% low risk (no)  → 22,603 samples
- 5.8%  high risk (yes) →  1,397 samples

The model would get lazy and predict "no" for everything,
giving a poor F1-score for the "yes" class.

**SMOTE (Synthetic Minority Oversampling Technique)** fixes this by
creating synthetic new "yes" samples — not copies, but new samples
generated based on existing "yes" patterns.

⚠️ SMOTE is applied ONLY on training data, never on test data.
Test data must reflect real world distribution.

In [20]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_sm.value_counts().to_dict())

Before SMOTE: {0: 22603, 1: 1397}
After SMOTE: {0: 22603, 1: 22603}


## Step 12: Feature Scaling
Logistic Regression converges faster and performs better when all 
features are on the same scale.

**StandardScaler** transforms each feature so that:
- Mean = 0
- Standard deviation = 1

For example:
- AGE ranges from 40-92 (large numbers)
- ORDER_HOUR ranges from 0-23 (small numbers)
- Without scaling, large numbers dominate the model

⚠️ Important rule:
- `fit_transform` on TRAINING data — learns the scale FROM training data
- `transform` only on TEST data — applies the same scale, never learns from test data

In [21]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled = scaler.transform(X_test)

## Step 13: Train Logistic Regression Model
Now we train the model using the scaled, balanced training data.
The model learns patterns from 45,206 balanced samples to predict
high risk vs low risk orders.

In [22]:
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train_sm)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

## Step 14: Model Evaluation
We evaluate the model on the **test data** (unseen data) using:
- `classification_report` — shows Precision, Recall and F1-score for each class
- `f1_score` — specifically measures performance on the "yes" (high risk) class

Our target: F1-score for "yes" class must be ≥ 70%

In [23]:
y_pred = model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))
print("F1-score for YES class:", f1_score(y_test, y_pred, pos_label=1))

              precision    recall  f1-score   support

           0       0.95      0.93      0.94      5651
           1       0.13      0.17      0.15       349

    accuracy                           0.88      6000
   macro avg       0.54      0.55      0.54      6000
weighted avg       0.90      0.88      0.89      6000

F1-score for YES class: 0.14532019704433496


## Step 15: Feature Selection
Not all 52 features contribute equally to predicting payment default.
We use correlation analysis to identify which features have the 
strongest relationship with the target variable (CLASS).

Removing weak features can:
- Reduce noise in the model
- Improve F1-score
- Make the model simpler and more interpretable

In [24]:
# Add target back temporarily to check correlations
X_corr = X.copy()
X_corr['CLASS'] = y

# Calculate correlation with target
correlations = X_corr.corr()['CLASS'].abs().sort_values(ascending=False)
print(correlations.head(20))

CLASS                    1.000000
NEUKUNDE_yes             0.098564
B_EMAIL_yes              0.085814
AMOUNT_ORDER_PRE         0.069607
CHK_LADR_yes             0.057418
SESSION_TIME             0.056533
B_TELEFON_yes            0.048338
Z_METHODE_debit_card     0.045803
Z_CARD_ART_debit_card    0.045803
CHK_COOKIE_yes           0.044273
WEEKDAY_ORDER_Sunday     0.040604
VALUE_ORDER_PRE          0.038892
FAIL_LORT_yes            0.036640
CHK_IP_yes               0.033601
CHK_RADR_yes             0.031751
FAIL_RORT_yes            0.029896
FLAG_NEWSLETTER_yes      0.028362
WEEKDAY_ORDER_Tuesday    0.019625
LAST_ORDER_YEAR          0.018670
Z_CARD_ART_Visa          0.017496
Name: CLASS, dtype: float64


In [25]:
# Select features with correlation above threshold
selected_features = correlations[correlations > 0.02].index.tolist()
selected_features.remove('CLASS')
print(f"Selected {len(selected_features)} features out of 52")
print(selected_features)

Selected 16 features out of 52
['NEUKUNDE_yes', 'B_EMAIL_yes', 'AMOUNT_ORDER_PRE', 'CHK_LADR_yes', 'SESSION_TIME', 'B_TELEFON_yes', 'Z_METHODE_debit_card', 'Z_CARD_ART_debit_card', 'CHK_COOKIE_yes', 'WEEKDAY_ORDER_Sunday', 'VALUE_ORDER_PRE', 'FAIL_LORT_yes', 'CHK_IP_yes', 'CHK_RADR_yes', 'FAIL_RORT_yes', 'FLAG_NEWSLETTER_yes']


In [26]:
# Filter to selected features only
X_train_sel = X_train_sm[selected_features]
X_test_sel = X_test[selected_features]

# Scale the selected features
scaler_sel = StandardScaler()
X_train_sel_scaled = scaler_sel.fit_transform(X_train_sel)
X_test_sel_scaled = scaler_sel.transform(X_test_sel)

# Retrain Logistic Regression
model_sel = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model_sel.fit(X_train_sel_scaled, y_train_sm)

# Evaluate
y_pred_sel = model_sel.predict(X_test_sel_scaled)
print(classification_report(y_test, y_pred_sel))
print("F1-score for YES class:", f1_score(y_test, y_pred_sel, pos_label=1))

              precision    recall  f1-score   support

           0       0.96      0.68      0.80      5651
           1       0.09      0.53      0.16       349

    accuracy                           0.67      6000
   macro avg       0.53      0.61      0.48      6000
weighted avg       0.91      0.67      0.76      6000

F1-score for YES class: 0.15920826161790017


## Step 16: Threshold Tuning
By default, Logistic Regression predicts "yes" when probability > 0.5.
We lower the threshold to 0.3 to make the model more sensitive to high risk orders.

- **Default (0.5):** Only predicts high risk when very confident
- **Tuned (0.3):** Predicts high risk with less confidence needed → catches more

This trades some precision for better recall on the "yes" class,
which is more important for the business — missing a high risk order
is more costly than a false alarm.



In [27]:
y_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred_tuned = (y_prob > 0.3).astype(int)
print(classification_report(y_test, y_pred_tuned))
print("F1-score for YES class (Threshold 0.3):", f1_score(y_test, y_pred_tuned, pos_label=1))

              precision    recall  f1-score   support

           0       0.96      0.77      0.86      5651
           1       0.11      0.45      0.18       349

    accuracy                           0.76      6000
   macro avg       0.53      0.61      0.52      6000
weighted avg       0.91      0.76      0.82      6000

F1-score for YES class (Threshold 0.3): 0.17557681485649973


## Step 17: Random Forest Classifier
Random Forest builds 100 decision trees and takes a majority vote.
It handles complex non-linear patterns that Logistic Regression cannot.
Does NOT require feature scaling.

Key parameters:
- `n_estimators=100` — builds 100 decision trees
- `class_weight='balanced'` — handles class imbalance internally
- `random_state=42` — reproducibility
- `n_jobs=-1` — uses all CPU cores for faster training

In [28]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Train on SMOTE balanced data
rf_model.fit(X_train_sm, y_train_sm)

# Use threshold tuning like we did for Logistic Regression
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]
y_pred_rf_tuned = (y_prob_rf > 0.3).astype(int)
print(classification_report(y_test, y_pred_rf_tuned))
print("F1-score for YES class (RF + Threshold 0.3):", f1_score(y_test, y_pred_rf_tuned, pos_label=1))

              precision    recall  f1-score   support

           0       0.95      0.93      0.94      5651
           1       0.14      0.17      0.16       349

    accuracy                           0.89      6000
   macro avg       0.54      0.55      0.55      6000
weighted avg       0.90      0.89      0.90      6000

F1-score for YES class (RF + Threshold 0.3): 0.1554140127388535


## Conclusion & Final Summary

### Project Overview
This project built a machine learning classification model to predict 
whether an online purchase order is at high risk (yes) or low risk (no) 
of payment default, using a dataset of 30,000 orders with 44 attributes.

---

### Preprocessing Pipeline
| Step | Action | Result |
|---|---|---|
| Step 3 | Dropped irrelevant columns | 44 → 41 columns |
| Step 4 | Filled numeric nulls with median | 0 missing values |
| Step 5 | Converted B_BIRTHDATE → AGE | Age range: 40-92 |
| Step 6 | Filled Z_CARD_ART with mode | 0 missing values |
| Step 7 | Converted TIME_ORDER → HOUR & MINUTE | 2 new features |
| Step 8 | Converted DATE_LORDER → YEAR/MONTH/DAY | 3 new features |
| Step 9 | One-hot encoded categorical columns | 41 → 52 columns |
| Step 10 | Stratified 80/20 train/test split | 24,000 / 6,000 |
| Step 11 | SMOTE balancing | 1,397 → 22,603 yes samples |
| Step 12 | StandardScaler normalization | Mean=0, Std=1 |

---

### Models & Results
| Model | Approach | F1 Score (YES class) |
|---|---|---|
| Logistic Regression | Baseline + SMOTE + Scaling | 14.5% |
| Logistic Regression | Feature Selection (16 features) | 15.9% |
| Logistic Regression | Threshold Tuning (0.3) | **17.5%** ← best |
| Random Forest | SMOTE + Threshold Tuning (0.3) | 15.5% |

---

### Why 70% F1 Was Not Achieved
1. **Weak feature correlations** — strongest correlation was only 0.099
2. **Severe class imbalance** — only 5.8% high risk orders
3. **Complex non-linear patterns** — beyond linear model capabilities

---

### Key Takeaway
All evaluations were performed on **unseen test data** to ensure 
honest and reliable performance metrics. Despite not reaching 70%, 
this project demonstrates thorough understanding of the full 
machine learning pipeline from raw data to model evaluation.